# DFIR-Guardrail: external dataset eval (phi3:mini)
Runs the pipeline **N_REPEATS times** under two conditions on `data/external.json` (500 artifacts: 250 benign WinLOLBIN-GT process events, 250 carrying third-party payloads from Lakera Gandalf and LLMail-Inject):

* **prefilter_on**: heuristic/semantic pre-filter gates the LLM (default pipeline)
* **model_only**: `--no_prefilter`, every artifact goes to the LLM

The dataset is identical every repeat, so variance is model non-determinism only. Pre-filter alone (offline, no LLM) scored 55.2% FNR and 16.8% FPR on this set.

**Expected runtime: about 30 minutes on a T4 with the defaults** (about 5 min setup, then about 2 min per prefilter_on run and about 5 min per model_only run, 3 repeats each). This is extrapolated from the synthetic ablation (about 0.86 s per LLM call with the pre-filter, about 0.58 s per call without), not measured on this dataset. The pre-filter forwards 154 of 500 artifacts. Finished runs are skipped on re-run.

Before running, build the bundle locally from the repo root and upload it when prompted:

```
python -c "import zipfile as z;f=z.ZipFile('external_bundle.zip','w',z.ZIP_DEFLATED);[f.write(p) for p in __import__('glob').glob('src/**/*.py',recursive=True)+['requirements.txt','data/external.json']]"
```

In [ ]:
# 1. Require a GPU, install Ollama, start the daemon
import subprocess
if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then re-run. CPU timeouts invalidate every result.")
!sudo apt-get update -y
!sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

import os, time, requests
os.environ["OLLAMA_NUM_PARALLEL"] = "3"   # must match CONCURRENCY below

def start_ollama():
    global ollama_process
    ollama_process = subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(60):
        try:
            if requests.get("http://localhost:11434").status_code == 200: return
        except Exception:
            time.sleep(1)
    raise RuntimeError("Ollama daemon did not become ready")

def restart_ollama():
    # Fresh daemon per run: requests that timed out in the previous run keep generating server-side and would starve the next one
    ollama_process.kill(); subprocess.run(["pkill", "-f", "ollama"]); time.sleep(3); start_ollama()

start_ollama()
!/usr/local/bin/ollama pull phi3:mini
# Preflight: one real generation, then confirm the model is resident on the GPU
t = time.time()
r = requests.post("http://localhost:11434/api/generate", json={"model": "phi3:mini", "prompt": "Say OK.", "stream": False}, timeout=300)
print(f"preflight generate: {time.time()-t:.1f}s", r.json().get("response", "")[:40])
ps = subprocess.run(["/usr/local/bin/ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
assert "100% GPU" in ps, "phi3:mini is not fully on the GPU; do not run the ablation"

In [ ]:
# 2. Install deps and unpack the exact local code
from google.colab import files
up = files.upload()  # choose external_bundle.zip
!unzip -o external_bundle.zip -d DFIR-Guardrail
%cd DFIR-Guardrail
!pip install -r requirements.txt

In [ ]:
# 3. Configuration
N_REPEATS = 3
CONCURRENCY = 3
DATASETS = {"external": "data/external.json"}
CONDITIONS = {"prefilter_on": ["--legacy_gate"], "gate_v2": [], "model_only": ["--no_prefilter"]}

In [ ]:
# 4. Run everything (resumable: a finished run's report is reused)
import json, subprocess, pathlib
runs = pathlib.Path("runs"); runs.mkdir(exist_ok=True)
rows = []
for ds, ds_file in DATASETS.items():
    for cond, extra in CONDITIONS.items():
        for rep in range(1, N_REPEATS + 1):
            out = runs / f"{ds}_{cond}_{rep}"
            report, log = out / "guardrail_simulation.json", out / "log.txt"
            if not report.exists():
                restart_ollama()
                out.mkdir(exist_ok=True)
                cmd = ["python", "src/pipeline.py", "--mode", "sim", "--concurrency", str(CONCURRENCY),
                       "--output_dir", str(out), *extra]
                cmd += ["--dataset_file", ds_file] if ds_file else ["--num_samples", str(NUM_SAMPLES), "--poison_ratio", str(POISON_RATIO)]
                print("RUN", ds, cond, rep, flush=True)
                with open(log, "w") as f:
                    subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env={**os.environ, "PYTHONPATH": "."})
            if not report.exists():
                print("  FAILED, see", log); continue
            m = json.load(open(report))["metadata"]
            rows.append({"dataset": ds, "condition": cond, "run": rep, "scan_s": m["scan_time_seconds"], **m["metrics"],
                         # >1% inference errors means the backend, not the model, produced the labels
                         "valid": m["metrics"]["Errors"] <= 0.01 * m["metrics"]["Total processed"]})
print("done:", len(rows), "runs")
# Record exactly what ran (model digest, Ollama version, prefilter/embedder status)
print(open(next(iter(runs.glob("*_1/log.txt")))).read().split("--- Run configuration ---")[-1][:600])

In [ ]:
# 5. Per-run table, then mean and min-max range per condition
import math, pandas as pd
df = pd.DataFrame(rows)
df["FNR"] = df["False Negative Rate (FNR)"] * 100
df["FPR"] = df["False Positive Rate (FPR)"] * 100
df.rename(columns={"True Positives":"TP","True Negatives":"TN","False Positives":"FP","False Negatives":"FN"}, inplace=True)
print(df[["dataset","condition","run","TP","TN","FP","FN","Errors","FNR","FPR","scan_s","valid"]].round(2).to_string(index=False))
bad = df[~df.valid]
if len(bad):
    print()
    print(f"EXCLUDED {len(bad)} invalid run(s) with >1% inference errors (timeouts are counted as flagged, which fakes detections):")
    print(bad[["dataset","condition","run","Errors"]].to_string(index=False))
df = df[df.valid]

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*2
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d; h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return 100*(c-h), 100*(c+h)

summary = []
for (ds, cond), g in df.groupby(["dataset","condition"], sort=False):
    pos, neg = int((g.TP+g.FN).iloc[0]), int((g.TN+g.FP).iloc[0])
    lo, hi = wilson(int(g.FN.sum()), int(pos*len(g)))   # pooled over repeats; repeats share one dataset, so treat as optimistic
    summary.append({"dataset":ds,"condition":cond,"runs":len(g),
        "FNR mean":g.FNR.mean(),"FNR min":g.FNR.min(),"FNR max":g.FNR.max(),
        "FPR mean":g.FPR.mean(),"FPR min":g.FPR.min(),"FPR max":g.FPR.max(),
        "scan_s mean":g.scan_s.mean(),"errors total":int(g.Errors.sum()),
        "FNR pooled CI lo":lo,"FNR pooled CI hi":hi})
summary = pd.DataFrame(summary).round(2)
print(summary.to_string(index=False))
summary.to_csv("external_summary.csv", index=False); df.to_csv("external_runs.csv", index=False)

In [ ]:
# 5b. Misses by payload source (per run, valid runs only)
import json
src_of = {d["id"]: d["source"] for d in json.load(open("data/external.json", encoding="utf-8"))}
rows2 = []
for r in df.itertuples():
    rep = json.load(open(f"runs/{r.dataset}_{r.condition}_{r.run}/guardrail_simulation.json"))["findings"]
    for f in rep:
        if f["ground_truth"] == "Malicious":
            rows2.append({"condition": r.condition, "run": r.run, "source": src_of[f["id"] - 1]  # report ids are 1-indexed, "miss": f["verdict"].startswith("Safe")})
by = pd.DataFrame(rows2).groupby(["condition", "source", "run"]).miss.mean().mul(100).groupby(["condition", "source"]).agg(["mean", "min", "max"]).round(1)
print("FNR % by payload source (125 artifacts each):"); print(by)
by.to_csv("external_fnr_by_source.csv")

In [ ]:
# 6. Download results and stop Ollama
import shutil
shutil.make_archive("external_results", "zip", ".", "runs")
for f in ["external_summary.csv", "external_runs.csv", "external_fnr_by_source.csv", "external_results.zip"]:
    files.download(f)
ollama_process.terminate()